# India Runs — Intelligent Candidate Discovery & Ranking System

**Team Name -** NoBlackBlox

---

## Install Dependencies

In [2]:
%%capture
!pip install rank_bm25 scikit-learn numpy pandas tqdm

## Step 1: Configuration

In [3]:
import json
import re
import math
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date, datetime
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')

CANDIDATES_PATH = "candidates.jsonl"   
TEAM_ID         = "NoBlackBox"          
OUTPUT_PATH     = "submission.csv"

TODAY = date.today()
print(f"Config ready. Output will be saved to: {OUTPUT_PATH}")

Config ready. Output will be saved to: submission.csv


## Step 2: Job Description Signals

These are derived directly from the JD by carefully reading what it *means*, not just what it *says*.

In [4]:
#  Read the JD carefully — keyword matching alone is a trap per the JD

JD_TEXT = """
Senior AI Engineer founding team Redrob AI Series A talent intelligence platform
Pune Noida India hybrid 5 to 9 years experience applied machine learning production
embeddings retrieval ranking LLM fine-tuning recommendation system search
sentence transformers BGE E5 OpenAI embeddings vector database hybrid search
Pinecone Weaviate Qdrant Milvus OpenSearch Elasticsearch FAISS dense retrieval
BM25 hybrid retrieval ranking evaluation NDCG MRR MAP A/B testing
python production code quality retrieval quality embedding drift index refresh
learning to rank XGBoost neural ranking LLM integration fine-tuning LoRA QLoRA PEFT
product company startup early stage NLP information retrieval recommendation
candidate job description matching talent acquisition recruiting platform
evaluation framework offline benchmark online experiment recruiter feedback
ship fast scrappy product engineering shipper not researcher
distributed systems large scale inference optimization open source contributions
"""

# Must-have hard skills — candidate must demonstrate these in career history
REQUIRED_SKILLS = [
    "embedding", "embeddings", "vector", "retrieval", "ranking", "search",
    "recommendation", "nlp", "information retrieval", "sentence transformer",
    "bge", "e5", "faiss", "pinecone", "weaviate", "qdrant", "milvus",
    "elasticsearch", "opensearch", "bm25", "dense retrieval", "hybrid search",
    "machine learning", "deep learning", "pytorch", "tensorflow", "transformers",
    "llm", "fine-tuning", "rag", "lora", "qlora", "peft",
    "ndcg", "mrr", "map", "a/b testing", "learning to rank", "xgboost",
    "python", "production ml", "applied ml"
]

# Nice-to-have signals
NICE_TO_HAVE_SKILLS = [
    "open source", "github", "distributed systems", "inference optimization",
    "hr tech", "recruiting", "talent", "marketplace", "lora", "qlora"
]

# Red flag title keywords — the JD explicitly disqualifies these profiles
RED_FLAG_TITLES = [
    "marketing manager", "hr manager", "content writer", "business analyst",
    "project manager", "product manager", "sales", "finance", "accountant",
    "computer vision", "speech", "robotics", "qa engineer", "tester"
]

# Consulting-only company signals (JD explicitly penalizes)
CONSULTING_FIRMS = [
    "tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini",
    "hcl", "tech mahindra", "mphasis", "hexaware", "niit"
]

# Preferred industries
GOOD_INDUSTRIES = [
    "technology", "software", "internet", "ai", "machine learning",
    "fintech", "edtech", "healthtech", "saas", "startup", "product",
    "e-commerce", "marketplace", "platform"
]

# Preferred locations (India-based)
PREFERRED_LOCATIONS = ["pune", "noida", "delhi", "ncr", "hyderabad", "mumbai", "bangalore", "bengaluru"]

# Ideal experience range per JD
YOE_IDEAL_MIN = 5
YOE_IDEAL_MAX = 9

print("JD signals configured.")
print(f"  Required skills: {len(REQUIRED_SKILLS)}")
print(f"  Nice-to-have: {len(NICE_TO_HAVE_SKILLS)}")
print(f"  Red flag titles: {len(RED_FLAG_TITLES)}")

JD signals configured.
  Required skills: 42
  Nice-to-have: 10
  Red flag titles: 14


## Step 3: Load & Validate Candidates

In [5]:
def load_candidates(path: str) -> list[dict]:
    """Load candidates.jsonl with basic validation."""
    candidates = []
    errors = 0
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                c = json.loads(line)
                if "candidate_id" not in c:
                    errors += 1
                    continue
                candidates.append(c)
            except json.JSONDecodeError:
                errors += 1
    print(f"Loaded {len(candidates):,} candidates ({errors} parse errors)")
    return candidates


t0 = time.time()
candidates = load_candidates(CANDIDATES_PATH)
print(f"Load time: {time.time()-t0:.1f}s")

Loaded 100,000 candidates (0 parse errors)
Load time: 5.6s


##  Step 4: Honeypot Detection

The dataset contains ~80 honeypots with *subtly impossible profiles*. We detect them using consistency checks — no special-casing, just data integrity validation.

In [6]:
# Checks for logically impossible / inconsistent profile signals.
def detect_honeypot(c: dict) -> tuple[bool, list[str]]:
    reasons = []
    profile  = c.get("profile", {})
    history  = c.get("career_history", [])
    skills   = c.get("skills", [])
    signals  = c.get("redrob_signals", {})

    # 1. Experience at a company that didn't exist long enough
    for job in history:
        try:
            start = date.fromisoformat(job["start_date"])
            duration_months = job.get("duration_months", 0)
            # If someone worked 8+ years at a company, its start must be before 2017
            if duration_months > 96 and start.year > 2018:
                reasons.append(f"Implausibly long tenure ({duration_months}mo) at {job.get('company','?')} starting {start}")
        except (ValueError, TypeError, KeyError):
            pass

    # 2. Expert skill with 0 months of use
    expert_zero = [s["name"] for s in skills
                   if s.get("proficiency") == "expert" and s.get("duration_months", 1) == 0]
    if len(expert_zero) >= 3:
        reasons.append(f"Expert proficiency in {len(expert_zero)} skills with 0 months use: {expert_zero[:3]}")

    # 3. Total YOE vs career history sum mismatch
    yoe = profile.get("years_of_experience", 0)
    history_months = sum(j.get("duration_months", 0) for j in history)
    history_years = history_months / 12
    if history_years > 0 and abs(history_years - yoe) > 5:
        reasons.append(f"YOE mismatch: profile says {yoe}y but history sums to {history_years:.1f}y")

    # 4. Skills declared but never appear in career descriptions
    expert_skills = [s["name"].lower() for s in skills if s.get("proficiency") == "expert"]
    all_desc = " ".join(j.get("description", "") for j in history).lower()
    missing_in_desc = [sk for sk in expert_skills if sk not in all_desc]
    if len(expert_skills) > 5 and len(missing_in_desc) == len(expert_skills):
        reasons.append(f"All {len(expert_skills)} expert skills absent from career descriptions")

    # 5. Implausible engagement: 0 last active but open to work
    try:
        last_active = date.fromisoformat(signals.get("last_active_date", "2000-01-01"))
        days_since = (TODAY - last_active).days
        if signals.get("open_to_work_flag") and days_since > 365:
            reasons.append(f"open_to_work=True but last active {days_since} days ago")
    except (ValueError, TypeError):
        pass

    # 6. Future dates in career history
    for job in history:
        try:
            start = date.fromisoformat(job["start_date"])
            if start > TODAY:
                reasons.append(f"Future start date in career history: {start}")
        except (ValueError, TypeError, KeyError):
            pass

    return len(reasons) > 0, reasons


honeypot_ids = set()
honeypot_reasons = {}
for c in tqdm(candidates, desc="Honeypot detection"):
    is_hp, reasons = detect_honeypot(c)
    if is_hp:
        honeypot_ids.add(c["candidate_id"])
        honeypot_reasons[c["candidate_id"]] = reasons

print(f"\nHoneypots detected: {len(honeypot_ids)}")
if honeypot_ids:
    sample_id = next(iter(honeypot_ids))
    print(f"Example — {sample_id}: {honeypot_reasons[sample_id]}")

Honeypot detection:   0%|          | 0/100000 [00:00<?, ?it/s]


Honeypots detected: 136
Example — CAND_0081852: ['All 6 expert skills absent from career descriptions']


## Step 5: Feature Engineering

#####  Helper functions for scoring candidates against the JD.

In [7]:
#  Convert text to lowercase.
def text_lower(text: str) -> str:
    return str(text).lower()

# Fraction of keywords found in text (0.0–1.0).
def keyword_hit_rate(text: str, keywords: list[str]) -> float:
    if not keywords or not text:
        return 0.0
    t = text_lower(text)
    hits = sum(1 for kw in keywords if kw in t)
    return hits / len(keywords)

# Build a single text blob from all profile signals for BM25/TF-IDF.
def build_candidate_text(c: dict) -> str:
    parts = []
    p = c.get("profile", {})
    parts.append(p.get("headline", ""))
    parts.append(p.get("summary", ""))
    parts.append(p.get("current_title", ""))
    parts.append(p.get("current_industry", ""))

    for job in c.get("career_history", []): 
        parts.append(job.get("title", ""))
        parts.append(job.get("description", ""))
        parts.append(job.get("company", ""))
        parts.append(job.get("industry", ""))

    for sk in c.get("skills", []):
        # Weight skills by proficiency — repeat expert skills more
        weight = {"expert": 4, "advanced": 3, "intermediate": 2, "beginner": 1}.get(sk.get("proficiency", "beginner"), 1)
        parts.extend([sk.get("name", "")] * weight)

    for cert in c.get("certifications", []):
        parts.append(cert.get("name", ""))

    return " ".join(parts) # Concatenate all parts into a single string


# Calculate days since a given date.
def days_since(date_str: str) -> int:
    try:
        d = date.fromisoformat(date_str)
        return (TODAY - d).days
    except (ValueError, TypeError):
        return 9999

# Score for years of experience vs JD ideal (5-9y).
def yoe_score(yoe: float) -> float:
    if yoe < 2:
        return 0.1
    if yoe < YOE_IDEAL_MIN:
        return 0.5 + 0.5 * (yoe - 2) / (YOE_IDEAL_MIN - 2)
    if yoe <= YOE_IDEAL_MAX:
        return 1.0
    if yoe <= 12:
        return 1.0 - 0.3 * (yoe - YOE_IDEAL_MAX) / (12 - YOE_IDEAL_MAX)
    return 0.4   # 12+ years: likely over-senior or title-chaser risk

# Location score based on preferred cities, country, and willingness to relocate.
def location_score(c: dict) -> float:
    loc = text_lower(c.get("profile", {}).get("location", ""))
    country = text_lower(c.get("profile", {}).get("country", ""))
    signals = c.get("redrob_signals", {})
    relocate = signals.get("willing_to_relocate", False)

    if any(city in loc for city in PREFERRED_LOCATIONS):
        return 1.0
    if country == "india" and relocate:
        return 0.75
    if country == "india":
        return 0.5
    if relocate:
        return 0.3
    return 0.1


#### Main feature extraction function — combines all signals into a single feature dict for scoring.

In [8]:
def extract_features(c: dict) -> dict:
    """
    Extract all scoring features for a single candidate.
    Returns a dict of named float features in [0, 1] (unless noted).
    """
    p         = c.get("profile", {})
    history   = c.get("career_history", [])
    skills    = c.get("skills", [])
    edu       = c.get("education", [])
    certs     = c.get("certifications", [])
    signals   = c.get("redrob_signals", {})
    cid       = c.get("candidate_id", "")

    full_text = build_candidate_text(c).lower()
    title_lc  = text_lower(p.get("current_title", ""))
    yoe       = float(p.get("years_of_experience", 0))

    feats = {}

    # 1.HARD DISQUALIFIERS — any red flag title is an automatic fail per JD
    red_flag_title = any(rf in title_lc for rf in RED_FLAG_TITLES)
    feats["red_flag_title"] = 1.0 if red_flag_title else 0.0

    # Consulting-only career (all jobs at big consulting firms)
    companies = [text_lower(j.get("company", "")) for j in history]
    consulting_count = sum(1 for co in companies if any(cf in co for cf in CONSULTING_FIRMS))
    feats["consulting_only"] = 1.0 if (len(companies) > 0 and consulting_count == len(companies)) else 0.0

    # No production ML experience (researcher-only signal)
    production_keywords = ["production", "deployed", "shipped", "launched", "release",
                           "real users", "live", "scale", "serving", "inference"]
    feats["has_production_ml"] = min(1.0, keyword_hit_rate(full_text, production_keywords) * 5)

    # Pure research background (academic only)
    research_titles = ["research scientist", "research engineer", "phd intern",
                       "postdoc", "research associate", "academic"]
    research_title_count = sum(1 for j in history
                               if any(rt in text_lower(j.get("title", "")) for rt in research_titles))
    feats["research_heavy"] = min(1.0, research_title_count / max(len(history), 1))

    # 2.SKILLS SIGNALS
    skill_names_lc = [text_lower(s.get("name", "")) for s in skills]

    # Required skills hit rate
    feats["required_skills_hit"] = keyword_hit_rate(" ".join(skill_names_lc), REQUIRED_SKILLS)

    # Nice-to-have skills
    feats["nice_skills_hit"] = keyword_hit_rate(full_text, NICE_TO_HAVE_SKILLS)

    # Advanced/Expert skill count (weighted)
    expert_count = sum(1 for s in skills if s.get("proficiency") in ("expert", "advanced"))
    feats["expert_skill_count_norm"] = min(1.0, expert_count / 10)

    # Embedding/retrieval specific deep skills
    retrieval_terms = ["embedding", "vector", "retrieval", "faiss", "pinecone",
                       "weaviate", "qdrant", "milvus", "elasticsearch", "bm25",
                       "dense retrieval", "hybrid search", "sentence transformer",
                       "semantic search", "ann", "approximate nearest neighbor"]
    feats["retrieval_depth"] = min(1.0, keyword_hit_rate(full_text, retrieval_terms) * 3)

    # Ranking/evaluation depth
    eval_terms = ["ndcg", "mrr", "map", "a/b test", "ab test", "learning to rank",
                  "ltr", "xgboost rank", "ranknet", "lambdamart", "evaluation", "benchmark"]
    feats["eval_depth"] = min(1.0, keyword_hit_rate(full_text, eval_terms) * 4)

    # LLM integration depth
    llm_terms = ["llm", "rag", "fine-tun", "lora", "qlora", "peft", "instruction tuning",
                 "gpt", "bert", "transformers", "huggingface", "langchain"]
    feats["llm_depth"] = min(1.0, keyword_hit_rate(full_text, llm_terms) * 3)

    # Python proficiency
    python_skills = [s for s in skills if "python" in text_lower(s.get("name", ""))]
    if python_skills:
        prof_map = {"expert": 1.0, "advanced": 0.8, "intermediate": 0.5, "beginner": 0.2}
        feats["python_strength"] = max(prof_map.get(s.get("proficiency", "beginner"), 0.1)
                                       for s in python_skills)
    else:
        feats["python_strength"] = 0.3 if "python" in full_text else 0.0

    # 3. EXPERIENCE SIGNALS
    feats["yoe_fit"] = yoe_score(yoe)

    # Applied ML tenure (% of career in ML/AI roles)
    ml_keywords = ["machine learning", "ml engineer", "ai engineer", "data scientist",
                   "nlp", "deep learning", "applied ml", "research engineer"]
    ml_months = sum(j.get("duration_months", 0) for j in history
                    if any(kw in text_lower(j.get("title", "") + " " + j.get("description", ""))
                           for kw in ml_keywords))
    total_months = sum(j.get("duration_months", 0) for j in history)
    feats["ml_career_fraction"] = ml_months / max(total_months, 1)

    # Product company experience (vs pure consulting/research)
    product_industries = ["technology", "software", "internet", "saas", "fintech",
                          "edtech", "healthtech", "e-commerce", "marketplace"]
    product_months = sum(j.get("duration_months", 0) for j in history
                         if any(pi in text_lower(j.get("industry", "")) for pi in product_industries))
    feats["product_company_fraction"] = product_months / max(total_months, 1)

    # Career tenure (short stints are a red flag per JD)
    if len(history) >= 2:
        avg_tenure = total_months / len(history)
        # Ideal: 18–48 months per role; short-hopping < 12 months per role is a red flag
        feats["tenure_stability"] = min(1.0, avg_tenure / 36) if avg_tenure < 36 else max(0.5, 1.0 - (avg_tenure - 48) / 100)
    else:
        feats["tenure_stability"] = 0.5

    # Current role recency and relevance
    current_jobs = [j for j in history if j.get("is_current")]
    if current_jobs:
        curr = current_jobs[0]
        curr_ml = any(kw in text_lower(curr.get("title", "") + " " + curr.get("description", ""))
                      for kw in ml_keywords)
        feats["current_role_ml"] = 1.0 if curr_ml else 0.2
    else:
        feats["current_role_ml"] = 0.3

    # Education tier (tier_1/tier_2 preferred)
    tier_map = {"tier_1": 1.0, "tier_2": 0.8, "tier_3": 0.6, "tier_4": 0.4, "unknown": 0.5}
    edu_score = max((tier_map.get(e.get("tier", "unknown"), 0.5) for e in edu), default=0.4)
    feats["edu_tier"] = edu_score

    # 4. BEHAVIORAL / ENGAGEMENT SIGNALS
    # Recency of activity (inactive candidates are poor fits even if qualified)
    last_active_days = days_since(signals.get("last_active_date", "2000-01-01"))
    if last_active_days < 7:    feats["recency"] = 1.0
    elif last_active_days < 30: feats["recency"] = 0.85
    elif last_active_days < 90: feats["recency"] = 0.6
    elif last_active_days < 180: feats["recency"] = 0.35
    else:                       feats["recency"] = 0.1

    # Open to work
    feats["open_to_work"] = 1.0 if signals.get("open_to_work_flag") else 0.4

    # Recruiter response rate (engagement signal)
    feats["recruiter_response"] = float(signals.get("recruiter_response_rate", 0.0))

    # Notice period — JD prefers < 30 days
    notice = int(signals.get("notice_period_days", 90))
    if notice <= 15:   feats["notice_score"] = 1.0
    elif notice <= 30: feats["notice_score"] = 0.9
    elif notice <= 60: feats["notice_score"] = 0.6
    elif notice <= 90: feats["notice_score"] = 0.4
    else:              feats["notice_score"] = 0.2

    # GitHub activity (open source contributions valued)
    gh = float(signals.get("github_activity_score", -1))
    feats["github_score"] = gh / 100 if gh >= 0 else 0.3  # -1 = no github linked

    # Profile completeness
    feats["completeness"] = float(signals.get("profile_completeness_score", 50)) / 100

    # Recruiter interest signals
    saved_30d = int(signals.get("saved_by_recruiters_30d", 0))
    feats["recruiter_interest"] = min(1.0, saved_30d / 10)

    # Interview completion rate (reliability)
    feats["interview_reliability"] = float(signals.get("interview_completion_rate", 0.5))

    # Work mode preference (JD is hybrid)
    pref = signals.get("preferred_work_mode", "flexible")
    mode_map = {"hybrid": 1.0, "flexible": 0.9, "onsite": 0.7, "remote": 0.5}
    feats["work_mode_fit"] = mode_map.get(pref, 0.6)

    # Location fit
    feats["location_fit"] = location_score(c)

    # Salary expectations (proxy for seniority/fit with startup budget)
    sal = signals.get("expected_salary_range_inr_lpa", {})
    sal_min = float(sal.get("min", 0))
    sal_max = float(sal.get("max", 999))
    sal_mid = (sal_min + sal_max) / 2
    # Reasonable range for 5-9 yr senior in India: 25-60 LPA
    if 20 <= sal_mid <= 70:    feats["salary_fit"] = 1.0
    elif 15 <= sal_mid < 20:   feats["salary_fit"] = 0.7
    elif 70 < sal_mid <= 100:  feats["salary_fit"] = 0.6
    else:                      feats["salary_fit"] = 0.3

    # Verified identity signals
    verified = (signals.get("verified_email") and signals.get("verified_phone"))
    feats["verified"] = 1.0 if verified else 0.5

    # 5. HONEYPOT PENALTY — JD explicitly warns against honeypots; apply a strong penalty if detected
    feats["is_honeypot"] = 1.0 if cid in honeypot_ids else 0.0

    return feats


In [9]:
t1 = time.time()

# Build feature matrix
feature_rows = []
candidate_texts = []
for c in tqdm(candidates, desc="Extracting features"):
    feats = extract_features(c)
    feats["candidate_id"] = c["candidate_id"]
    feature_rows.append(feats)
    candidate_texts.append(build_candidate_text(c))

feat_df = pd.DataFrame(feature_rows).set_index("candidate_id")
print(f"Feature extraction done in {time.time()-t1:.1f}s")
print(f"Feature shape: {feat_df.shape}")
feat_df.head(3)

Extracting features:   0%|          | 0/100000 [00:00<?, ?it/s]

Feature extraction done in 37.7s
Feature shape: (100000, 30)


,red_flag_title,consulting_only,has_production_ml,research_heavy,required_skills_hit,nice_skills_hit,expert_skill_count_norm,retrieval_depth,eval_depth,llm_depth,...,notice_score,github_score,completeness,recruiter_interest,interview_reliability,work_mode_fit,location_fit,salary_fit,verified,is_honeypot
candidate_id,,,,,,,,,,,,,,,,,,,,,
CAND_0000001,0.0,0.0,0.0,0.0,0.119048,0.1,0.7,0.1875,0.0,1.00,...,0.6,0.092,0.869,0.4,0.71,0.7,0.1,1.0,1.0,0.0
CAND_0000002,0.0,0.0,1.0,0.0,0.000000,0.0,0.0,0.0000,0.0,0.50,...,0.6,0.300,0.787,1.0,0.62,0.9,0.5,0.3,0.5,0.0
CAND_0000003,0.0,1.0,0.0,0.0,0.000000,0.0,0.0,0.0000,0.0,0.25,...,0.2,0.300,0.319,0.4,0.86,1.0,0.1,0.3,0.5,0.0


##  Step 6: Semantic Scoring (TF-IDF + BM25)

We use two complementary approaches:
- **TF-IDF cosine similarity**: captures term importance across the corpus
- **BM25**: classic probabilistic retrieval, good for keyword precision


In [10]:
t2 = time.time()

# TF-IDF vectorization for semantic similarity 
tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"(?u)\b[a-z][a-z+#/\.]{1,}\b"
)

# Fit on all candidate texts
candidate_tfidf = tfidf.fit_transform(candidate_texts)

# JD vector
jd_vec = tfidf.transform([JD_TEXT])
tfidf_scores = cosine_similarity(jd_vec, candidate_tfidf).flatten()

# BM25 scoring for semantic similarity 
tokenized = [text.lower().split() for text in candidate_texts]
bm25 = BM25Okapi(tokenized)
jd_tokens = JD_TEXT.lower().split()
bm25_scores = bm25.get_scores(jd_tokens)

# Normalize BM25 to [0, 1]
bm25_max = bm25_scores.max()
bm25_norm = bm25_scores / bm25_max if bm25_max > 0 else bm25_scores

# Combined semantic score (TF-IDF + BM25 ensemble)
semantic_scores = 0.6 * tfidf_scores + 0.4 * bm25_norm

# Map to candidate_id
cand_ids = [c["candidate_id"] for c in candidates]
semantic_df = pd.DataFrame({
    "candidate_id": cand_ids,
    "tfidf_score": tfidf_scores,
    "bm25_score": bm25_norm,
    "semantic_score": semantic_scores
}).set_index("candidate_id")

print(f"Semantic scoring done in {time.time()-t2:.1f}s")
print(semantic_df.describe())

Semantic scoring done in 119.1s
         tfidf_score     bm25_score  semantic_score
count  100000.000000  100000.000000   100000.000000
mean        0.022192       0.118812        0.060840
std         0.018701       0.075913        0.041239
min         0.003459       0.046858        0.021052
25%         0.011990       0.077817        0.038493
50%         0.015971       0.090182        0.045477
75%         0.022644       0.125419        0.063790
max         0.253727       1.000000        0.546146


##  Step 7: Composite Score & Ranking

Weights are designed based on the JD's explicit priorities:
- Semantic fit is necessary but not sufficient
- Hard skills (retrieval, evaluation depth) are the highest signal
- Behavioral signals (activity, response rate) are critical for actually hiring
- Location and logistics matter for a hybrid India role

In [11]:
# Score weights for each feature — these are tuned to reflect the JD's priorities and the relative importance of each signal.

WEIGHTS = {
    # Semantic / text relevance
    "semantic_score":         0.12,

    # Hard technical requirements
    "retrieval_depth":        0.10,
    "required_skills_hit":    0.08,
    "eval_depth":             0.07,
    "llm_depth":              0.05,
    "python_strength":        0.04,

    # Experience quality
    "yoe_fit":                0.06,
    "ml_career_fraction":     0.05,
    "product_company_fraction":0.05,
    "current_role_ml":        0.04,
    "has_production_ml":      0.04,
    "tenure_stability":       0.02,

    # Engagement / behavioral
    "recency":                0.05,
    "open_to_work":           0.03,
    "recruiter_response":     0.03,
    "notice_score":           0.02,
    "interview_reliability":  0.02,
    "recruiter_interest":     0.02,

    # Nice-to-have
    "github_score":           0.02,
    "nice_skills_hit":        0.01,
    "edu_tier":               0.01,

    # Logistics
    "location_fit":           0.03,
    "work_mode_fit":          0.01,
    "salary_fit":             0.01,
    "completeness":           0.01,
    "verified":               0.01,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS.values()):.3f}, must be 1.0"

# penalty factors for disqualifying signals — these are multiplicative penalties applied to the raw score.
PENALTIES = {
    "red_flag_title":    0.80,   # Very strong penalty — wrong career domain
    "consulting_only":   0.50,   # Strong penalty — explicit JD signal
    "research_heavy":    0.40,   # Moderate penalty — JD says "shipper not researcher"
    "is_honeypot":       0.95,   # Near-total penalty
}

# Merge all feature dataframes
all_df = feat_df.join(semantic_df, how="left")

# Compute raw weighted score
score_cols = [col for col in WEIGHTS if col in all_df.columns]
raw_score = sum(all_df[col].fillna(0) * w for col, w in WEIGHTS.items() if col in all_df.columns)

# Apply penalties (multiplicative)
penalty_mult = pd.Series(1.0, index=all_df.index)
for pen_col, pen_factor in PENALTIES.items():
    if pen_col in all_df.columns:
        penalty_mult = penalty_mult * (1 - all_df[pen_col].fillna(0) * pen_factor)

composite = raw_score * penalty_mult

# Normalize to [0, 1]
c_min, c_max = composite.min(), composite.max()
if c_max > c_min:
    composite_norm = (composite - c_min) / (c_max - c_min)
else:
    composite_norm = composite.copy()

all_df["composite_score"] = composite_norm

# Sort: descending score, tie-break by candidate_id ascending
ranked = all_df.sort_values(
    ["composite_score", "candidate_id"],
    ascending=[False, True]
).reset_index()

# Take top 100
top100 = ranked.head(100).copy()
top100["rank"] = range(1, 101)

# Verify scores are non-increasing 
assert all(top100["composite_score"].diff().dropna() <= 1e-9), "Scores not non-increasing!"

print(f"Score range in top-100: {top100['composite_score'].min():.4f} – {top100['composite_score'].max():.4f}")
print(f"Honeypots in top-100: {top100['is_honeypot'].sum():.0f}")
print(top100[["candidate_id", "rank", "composite_score"]].head(10))

Score range in top-100: 0.7513 – 1.0000
Honeypots in top-100: 0
   candidate_id  rank  composite_score
0  CAND_0081846     1         1.000000
1  CAND_0077337     2         0.968324
2  CAND_0046064     3         0.964328
3  CAND_0088025     4         0.950351
4  CAND_0018499     5         0.944648
5  CAND_0002025     6         0.930053
6  CAND_0046525     7         0.918778
7  CAND_0008425     8         0.915268
8  CAND_0071974     9         0.903891
9  CAND_0037944    10         0.897890


##  Step 8: Reasoning Generation

Generates specific, fact-grounded reasoning per the Stage 4 rubric:
- Specific facts from the profile
- JD connection
- Honest about gaps
- No hallucination
- Rank-consistent tone

In [12]:
# Build specific, honest reasoning for each candidate in top-100
def build_reasoning(c: dict, feats: dict, rank: int, score: float) -> str:
    p       = c.get("profile", {})
    signals = c.get("redrob_signals", {})
    history = c.get("career_history", [])
    skills  = c.get("skills", [])

    title   = p.get("current_title", "Engineer")
    yoe     = p.get("years_of_experience", 0)
    company = p.get("current_company", "")
    loc     = p.get("location", "")

    # Top skills (expert/advanced, most endorsed)
    top_skills = sorted(
        [s for s in skills if s.get("proficiency") in ("expert", "advanced")],
        key=lambda s: s.get("endorsements", 0), reverse=True
    )[:3]
    top_skill_names = ", ".join(s["name"] for s in top_skills) if top_skills else "general ML"

    # Notice period
    notice = signals.get("notice_period_days", 90)

    # Response rate
    rr = signals.get("recruiter_response_rate", 0)

    # Last active
    last_active_days = days_since(signals.get("last_active_date", "2000-01-01"))

    # Build strengths and concerns
    strengths = []
    concerns  = []

    if feats.get("retrieval_depth", 0) > 0.5:
        strengths.append("strong retrieval/embedding background")
    if feats.get("eval_depth", 0) > 0.3:
        strengths.append("ranking evaluation experience (NDCG/MRR)")
    if feats.get("product_company_fraction", 0) > 0.6:
        strengths.append("product-company career")
    if feats.get("has_production_ml", 0) > 0.5:
        strengths.append("production ML deployment")
    if feats.get("github_score", 0) > 0.5:
        strengths.append("active open-source contributions")
    if feats.get("llm_depth", 0) > 0.4:
        strengths.append("LLM/fine-tuning experience")

    if notice > 60:
        concerns.append(f"long notice period ({notice}d)")
    if rr < 0.3:
        concerns.append(f"low recruiter response rate ({rr:.0%})")
    if last_active_days > 90:
        concerns.append(f"inactive for {last_active_days} days")
    if feats.get("red_flag_title", 0) > 0:
        concerns.append(f"current title ({title}) is outside AI/ML domain")
    if feats.get("consulting_only", 0) > 0:
        concerns.append("career exclusively at consulting firms")
    if feats.get("research_heavy", 0) > 0.5:
        concerns.append("research-heavy background without clear production deployments")
    if feats.get("is_honeypot", 0) > 0:
        concerns.append("profile has data inconsistencies (honeypot signal)")

    # Build sentence 1: core profile
    s1 = f"{yoe:.0f}-year {title} at {company}" if company else f"{yoe:.0f}-year {title}"
    if loc:
        s1 += f", {loc}"
    if top_skills:
        s1 += f"; top skills: {top_skill_names}"

    # Build sentence 2: strengths + concerns balanced by rank
    parts2 = []
    if rank <= 10:
        # Emphasize strengths, mention any real concerns
        if strengths:
            parts2.append("Strong fit: " + ", ".join(strengths[:3]))
        if concerns:
            parts2.append("Concern: " + "; ".join(concerns[:1]))
    elif rank <= 30:
        if strengths:
            parts2.append("Fit signals: " + ", ".join(strengths[:2]))
        if concerns:
            parts2.append("Gap: " + "; ".join(concerns[:2]))
    elif rank <= 60:
        if strengths:
            parts2.append("Partial match: " + ", ".join(strengths[:1]))
        if concerns:
            parts2.append("Notable gaps: " + "; ".join(concerns[:2]))
    else:
        # Low rank — be honest about weaknesses
        if concerns:
            parts2.append("Significant gaps: " + "; ".join(concerns[:3]))
        elif strengths:
            parts2.append("Adjacent skills only — weak JD fit overall")
        else:
            parts2.append("Profile lacks core JD requirements")

    if not parts2:
        parts2.append("Limited alignment with JD requirements")

    s2 = "; ".join(parts2)
    return f"{s1}. {s2}."


# Build a lookup map from candidate_id -> (candidate dict, features dict)
cand_map = {c["candidate_id"]: c for c in candidates}
feat_map = {row["candidate_id"]: row.to_dict() for _, row in feat_df.reset_index().iterrows()}


In [13]:
t3 = time.time()
reasonings = []
for _, row in tqdm(top100.iterrows(), total=100, desc="Generating reasonings"):
    cid   = row["candidate_id"]
    rank  = int(row["rank"])
    score = float(row["composite_score"])
    c     = cand_map.get(cid, {})
    feats = feat_map.get(cid, {})
    r     = build_reasoning(c, feats, rank, score)
    reasonings.append(r)

top100["reasoning"] = reasonings
print(f"Reasoning generation done in {time.time()-t3:.1f}s")

# Spot-check
for i in [0, 4, 9, 49, 99]:
    row = top100.iloc[i]
    print(f"\nRank {row['rank']} | Score {row['composite_score']:.4f}")
    print(f"  {row['reasoning']}")

Generating reasonings:   0%|          | 0/100 [00:00<?, ?it/s]

Reasoning generation done in 0.1s

Rank 1 | Score 1.0000
  7-year Lead AI Engineer at Razorpay, Jaipur, Rajasthan; top skills: Information Retrieval, Learning to Rank, Vector Search. Strong fit: strong retrieval/embedding background, ranking evaluation experience (NDCG/MRR), product-company career.

Rank 5 | Score 0.9446
  7-year Senior Machine Learning Engineer at Zomato, Noida, Uttar Pradesh; top skills: scikit-learn, Recommendation Systems, Learning to Rank. Strong fit: strong retrieval/embedding background, ranking evaluation experience (NDCG/MRR), product-company career.

Rank 10 | Score 0.8979
  5-year Senior Data Scientist at Vedantu, Jaipur, Rajasthan; top skills: ASR, QLoRA, LlamaIndex. Strong fit: strong retrieval/embedding background, ranking evaluation experience (NDCG/MRR), product-company career.

Rank 50 | Score 0.8017
  8-year Senior Data Scientist at Amazon, Chennai, Tamil Nadu; top skills: PEFT, Machine Learning, Elasticsearch. Partial match: strong retrieval/embeddin

## Step 9: Validation (Pre-flight Checks)

Run all checks from Section 3 and Section 6 of the submission spec.

In [14]:
def validate_submission(df: pd.DataFrame, all_candidate_ids: set) -> bool:
    errors = []
    warnings_list = []

    # 1. Exactly 100 rows
    if len(df) != 100:
        errors.append(f"Row count = {len(df)}, expected 100")

    # 2. Ranks 1–100 each exactly once
    ranks = sorted(df["rank"].tolist())
    if ranks != list(range(1, 101)):
        errors.append(f"Ranks are not exactly 1–100 (got {len(set(ranks))} unique, min={min(ranks)}, max={max(ranks)})")

    # 3. Unique candidate_ids
    if df["candidate_id"].nunique() != 100:
        errors.append(f"Duplicate candidate_ids detected")

    # 4. All candidate_ids exist in the dataset
    unknown = set(df["candidate_id"]) - all_candidate_ids
    if unknown:
        errors.append(f"Unknown candidate_ids: {unknown}")

    # 5. Required columns in correct order
    req_cols = ["candidate_id", "rank", "score", "reasoning"]
    for col in ["candidate_id", "rank", "score"]:
        if col not in df.columns:
            errors.append(f"Missing required column: {col}")
    # Verify exact column order
    actual_cols = list(df.columns)
    if actual_cols[:4] != req_cols:
        errors.append(f"Column order wrong: got {actual_cols}, expected {req_cols}")

    # 6. Scores non-increasing with rank
    sorted_by_rank = df.sort_values("rank")
    score_diffs = sorted_by_rank["score"].diff().dropna()
    if (score_diffs > 1e-9).any():
        n_violations = (score_diffs > 1e-9).sum()
        errors.append(f"Scores not non-increasing: {n_violations} violation(s)")

    # 7. No all-same scores
    if df["score"].nunique() == 1:
        errors.append("All scores are identical — model isn't differentiating")

    # 8. Honeypots not in top 10
    top10_honeypots = df[df["rank"] <= 10]["candidate_id"].apply(
        lambda x: x in honeypot_ids).sum()
    if top10_honeypots > 0:
        warnings_list.append(f"{top10_honeypots} honeypots in top 10 — risk of Stage 3 disqualification")

    # 9. Reasoning not empty / not all identical
    if "reasoning" in df.columns:
        empty_reasoning = df["reasoning"].isna().sum() + (df["reasoning"] == "").sum()
        if empty_reasoning > 10:
            warnings_list.append(f"{empty_reasoning} empty reasoning entries")
        if df["reasoning"].nunique() < 50:
            warnings_list.append(f"Low reasoning diversity: only {df['reasoning'].nunique()} unique values")

    # Report
    
    if not errors:
        print("✅ ALL VALIDATION CHECKS PASSED")
    else:
        print(f"❌ {len(errors)} ERROR(S) FOUND:")
        for e in errors:
            print(f"   • {e}")

    if warnings_list:
        print(f"\n⚠️  {len(warnings_list)} WARNING(S):")
        for w in warnings_list:
            print(f"   • {w}")
    
    return len(errors) == 0


all_ids = {c["candidate_id"] for c in candidates}

# Build the submission DataFrame
submission = top100[["candidate_id", "rank", "composite_score", "reasoning"]].copy()
submission = submission.rename(columns={"composite_score": "score"})

# Ensure scores are non-increasing (resolve floating point ties)
submission = submission.sort_values("rank")
scores = submission["score"].values.copy()
for i in range(1, len(scores)):
    if scores[i] > scores[i-1]:
        scores[i] = scores[i-1]
submission["score"] = scores

is_valid = validate_submission(submission, all_ids)

✅ ALL VALIDATION CHECKS PASSED


##  Step 10: Save Submission

In [15]:
if is_valid:
    submission.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"✅ Submission saved to: {OUTPUT_PATH}")
    print(f"   Rows: {len(submission)}")
    print(f"   Score range: {submission['score'].min():.4f} – {submission['score'].max():.4f}")
    print(f"   Unique candidate_ids: {submission['candidate_id'].nunique()}")
    print()
    print("Preview (top 5):")
    print(submission.head(5))
else:
    print("❌ Submission NOT saved — fix errors above first.")

✅ Submission saved to: submission.csv
   Rows: 100
   Score range: 0.7513 – 1.0000
   Unique candidate_ids: 100

Preview (top 5):
   candidate_id  rank     score  \
0  CAND_0081846     1  1.000000   
1  CAND_0077337     2  0.968324   
2  CAND_0046064     3  0.964328   
3  CAND_0088025     4  0.950351   
4  CAND_0018499     5  0.944648   

                                           reasoning  
0  7-year Lead AI Engineer at Razorpay, Jaipur, R...  
1  7-year Staff Machine Learning Engineer at Payt...  
2  9-year Senior NLP Engineer at Salesforce, Coim...  
3  9-year Staff Machine Learning Engineer at Yell...  
4  7-year Senior Machine Learning Engineer at Zom...  


##  Step 11: Download (Google Colab)

In [16]:
try:
    from google.colab import files
    if is_valid:
        files.download(OUTPUT_PATH)
        print("Download triggered.")
except ImportError:
    print(f"Not in Colab. Your file is at: {Path(OUTPUT_PATH).resolve()}")

Not in Colab. Your file is at: C:\Users\mahta\Desktop\IndiaRuns\submission.csv


##  Step 12: Analysis & Debugging

In [17]:
# ── Score distribution ────────────────────────────────────────────────────
print("=== Top 20 Candidates ===")
display_cols = ["candidate_id", "rank", "score",
                "retrieval_depth", "yoe_fit", "recency", "open_to_work", "reasoning"]
avail_cols = [c for c in display_cols if c in submission.columns or c in top100.columns]

debug_df = top100.head(20)[["candidate_id", "rank", "composite_score",
                              "retrieval_depth", "yoe_fit", "recency",
                              "open_to_work", "reasoning"]]
pd.set_option("display.max_colwidth", 80)
print(debug_df.to_string(index=False))

print("\n=== Feature Distributions (top-100) ===")
feature_cols = ["retrieval_depth", "eval_depth", "llm_depth",
                "yoe_fit", "ml_career_fraction", "product_company_fraction",
                "recency", "recruiter_response", "notice_score", "location_fit"]
avail_feats = [f for f in feature_cols if f in top100.columns]
print(top100[avail_feats].describe().round(3))

print("\n=== Penalty Statistics ===")
pen_cols = ["red_flag_title", "consulting_only", "research_heavy", "is_honeypot"]
avail_pens = [p for p in pen_cols if p in top100.columns]
print(top100[avail_pens].sum())

=== Top 20 Candidates ===
candidate_id  rank  composite_score  retrieval_depth  yoe_fit  recency  open_to_work                                                                                                                                                                                                                                                                                reasoning
CAND_0081846     1         1.000000           1.0000 1.000000     0.60           1.0                                         7-year Lead AI Engineer at Razorpay, Jaipur, Rajasthan; top skills: Information Retrieval, Learning to Rank, Vector Search. Strong fit: strong retrieval/embedding background, ranking evaluation experience (NDCG/MRR), product-company career.
CAND_0077337     2         0.968324           1.0000 1.000000     0.85           1.0                                              7-year Staff Machine Learning Engineer at Paytm, Kochi, Kerala; top skills: LlamaIndex, LLMs, Information Retrie

In [18]:
total_time = time.time() - t0
print(f"\n Total pipeline time: {total_time:.1f}s ({total_time/60:.2f} min)")
print(f"   Limit: 5 minutes | Status: {'✅ WITHIN LIMIT' if total_time < 300 else '❌ OVER LIMIT'}")


 Total pipeline time: 181.2s (3.02 min)
   Limit: 5 minutes | Status: ✅ WITHIN LIMIT


##  Step 13: One-Command Reproduction

For your `README.md`, the command to reproduce submission from scratch:

```bash
pip install rank_bm25 scikit-learn numpy pandas tqdm
python rank.py --candidates ./candidates.jsonl --team-id NoBlackBox --out ./NoBlackBox.csv
```

The cell below writes `rank.py` as a standalone script equivalent to this notebook.

In [19]:
RANK_SCRIPT = '''
#!/usr/bin/env python3
"""
rank.py — Redrob Hackathon Candidate Ranking
Usage:
    python rank.py --candidates candidates.jsonl --team-id NoBlackBox --out NoBlackBox.csv
"""
import argparse
import json
import time
import warnings
import numpy as np
import pandas as pd
from datetime import date
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

warnings.filterwarnings("ignore")
TODAY = date.today()

# ── JD Signals ──────────────────────────────────────────────────────────────
JD_TEXT = """
Senior AI Engineer founding team Redrob AI Series A talent intelligence platform
Pune Noida India hybrid 5 to 9 years experience applied machine learning production
embeddings retrieval ranking LLM fine-tuning recommendation system search
sentence transformers BGE E5 OpenAI embeddings vector database hybrid search
Pinecone Weaviate Qdrant Milvus OpenSearch Elasticsearch FAISS dense retrieval
BM25 hybrid retrieval ranking evaluation NDCG MRR MAP A/B testing
python production code quality retrieval quality embedding drift index refresh
learning to rank XGBoost neural ranking LLM integration fine-tuning LoRA QLoRA PEFT
product company startup early stage NLP information retrieval recommendation
candidate job description matching talent acquisition recruiting platform
evaluation framework offline benchmark online experiment recruiter feedback
ship fast scrappy product engineering shipper not researcher
distributed systems large scale inference optimization open source contributions
"""

REQUIRED_SKILLS = [
    "embedding", "embeddings", "vector", "retrieval", "ranking", "search",
    "recommendation", "nlp", "information retrieval", "sentence transformer",
    "bge", "e5", "faiss", "pinecone", "weaviate", "qdrant", "milvus",
    "elasticsearch", "opensearch", "bm25", "dense retrieval", "hybrid search",
    "machine learning", "deep learning", "pytorch", "tensorflow", "transformers",
    "llm", "fine-tuning", "rag", "lora", "qlora", "peft",
    "ndcg", "mrr", "map", "a/b testing", "learning to rank", "xgboost",
    "python", "production ml", "applied ml"
]

NICE_TO_HAVE_SKILLS = [
    "open source", "github", "distributed systems", "inference optimization",
    "hr tech", "recruiting", "talent", "marketplace", "lora", "qlora"
]

RED_FLAG_TITLES = [
    "marketing manager", "hr manager", "content writer", "business analyst",
    "project manager", "product manager", "sales", "finance", "accountant",
    "computer vision", "speech", "robotics", "qa engineer", "tester"
]

CONSULTING_FIRMS = [
    "tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini",
    "hcl", "tech mahindra", "mphasis", "hexaware", "niit"
]

PREFERRED_LOCATIONS = ["pune", "noida", "delhi", "ncr", "hyderabad", "mumbai", "bangalore", "bengaluru"]
YOE_IDEAL_MIN = 5
YOE_IDEAL_MAX = 9

WEIGHTS = {
    "semantic_score":          0.12,
    "retrieval_depth":         0.10,
    "required_skills_hit":     0.08,
    "eval_depth":              0.07,
    "llm_depth":               0.05,
    "python_strength":         0.04,
    "yoe_fit":                 0.06,
    "ml_career_fraction":      0.05,
    "product_company_fraction": 0.05,
    "current_role_ml":         0.04,
    "has_production_ml":       0.04,
    "tenure_stability":        0.02,
    "recency":                 0.05,
    "open_to_work":            0.03,
    "recruiter_response":      0.03,
    "notice_score":            0.02,
    "interview_reliability":   0.02,
    "recruiter_interest":      0.02,
    "github_score":            0.02,
    "nice_skills_hit":         0.01,
    "edu_tier":                0.01,
    "location_fit":            0.03,
    "work_mode_fit":           0.01,
    "salary_fit":              0.01,
    "completeness":            0.01,
    "verified":                0.01,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS.values()):.3f}"

PENALTIES = {
    "red_flag_title":  0.80,
    "consulting_only": 0.50,
    "research_heavy":  0.40,
    "is_honeypot":     0.95,
}

# ── Helpers ──────────────────────────────────────────────────────────────────
def text_lower(text: str) -> str:
    return str(text).lower()

def keyword_hit_rate(text: str, keywords: list) -> float:
    if not keywords or not text:
        return 0.0
    t = text_lower(text)
    return sum(1 for kw in keywords if kw in t) / len(keywords)

def days_since(date_str: str) -> int:
    try:
        return (TODAY - date.fromisoformat(date_str)).days
    except (ValueError, TypeError):
        return 9999

def yoe_score(yoe: float) -> float:
    if yoe < 2:            return 0.1
    if yoe < YOE_IDEAL_MIN: return 0.5 + 0.5 * (yoe - 2) / (YOE_IDEAL_MIN - 2)
    if yoe <= YOE_IDEAL_MAX: return 1.0
    if yoe <= 12:          return 1.0 - 0.3 * (yoe - YOE_IDEAL_MAX) / (12 - YOE_IDEAL_MAX)
    return 0.4

def location_score(c: dict) -> float:
    loc     = text_lower(c.get("profile", {}).get("location", ""))
    country = text_lower(c.get("profile", {}).get("country", ""))
    signals = c.get("redrob_signals", {})
    relocate = signals.get("willing_to_relocate", False)
    if any(city in loc for city in PREFERRED_LOCATIONS): return 1.0
    if country == "india" and relocate:                  return 0.75
    if country == "india":                               return 0.5
    if relocate:                                         return 0.3
    return 0.1

def build_candidate_text(c: dict) -> str:
    parts = []
    p = c.get("profile", {})
    parts += [p.get("headline", ""), p.get("summary", ""),
              p.get("current_title", ""), p.get("current_industry", "")]
    for job in c.get("career_history", []):
        parts += [job.get("title", ""), job.get("description", ""),
                  job.get("company", ""), job.get("industry", "")]
    for sk in c.get("skills", []):
        weight = {"expert": 4, "advanced": 3, "intermediate": 2, "beginner": 1}.get(
            sk.get("proficiency", "beginner"), 1)
        parts.extend([sk.get("name", "")] * weight)
    for cert in c.get("certifications", []):
        parts.append(cert.get("name", ""))
    return " ".join(parts)

# ── Honeypot Detection ───────────────────────────────────────────────────────
def detect_honeypot(c: dict, honeypot_ids: set) -> tuple:
    reasons = []
    profile  = c.get("profile", {})
    history  = c.get("career_history", [])
    skills   = c.get("skills", [])
    signals  = c.get("redrob_signals", {})

    for job in history:
        try:
            start = date.fromisoformat(job["start_date"])
            duration_months = job.get("duration_months", 0)
            if duration_months > 96 and start.year > 2018:
                reasons.append(f"Implausibly long tenure ({duration_months}mo) starting {start}")
        except (ValueError, TypeError, KeyError):
            pass

    expert_zero = [s["name"] for s in skills
                   if s.get("proficiency") == "expert" and s.get("duration_months", 1) == 0]
    if len(expert_zero) >= 3:
        reasons.append(f"Expert proficiency in {len(expert_zero)} skills with 0 months use")

    yoe = profile.get("years_of_experience", 0)
    history_months = sum(j.get("duration_months", 0) for j in history)
    if history_months > 0 and abs(history_months / 12 - yoe) > 5:
        reasons.append(f"YOE mismatch: profile={yoe}y, history={history_months/12:.1f}y")

    expert_skills = [s["name"].lower() for s in skills if s.get("proficiency") == "expert"]
    all_desc = " ".join(j.get("description", "") for j in history).lower()
    if len(expert_skills) > 5 and all(sk not in all_desc for sk in expert_skills):
        reasons.append(f"All {len(expert_skills)} expert skills absent from descriptions")

    try:
        last_active = date.fromisoformat(signals.get("last_active_date", "2000-01-01"))
        if signals.get("open_to_work_flag") and (TODAY - last_active).days > 365:
            reasons.append("open_to_work=True but inactive >365 days")
    except (ValueError, TypeError):
        pass

    for job in history:
        try:
            if date.fromisoformat(job["start_date"]) > TODAY:
                reasons.append(f"Future start date: {job['start_date']}")
        except (ValueError, TypeError, KeyError):
            pass

    return len(reasons) > 0, reasons

# ── Feature Extraction ───────────────────────────────────────────────────────
def extract_features(c: dict, honeypot_ids: set) -> dict:
    p       = c.get("profile", {})
    history = c.get("career_history", [])
    skills  = c.get("skills", [])
    edu     = c.get("education", [])
    signals = c.get("redrob_signals", {})
    cid     = c.get("candidate_id", "")

    full_text = build_candidate_text(c).lower()
    title_lc  = text_lower(p.get("current_title", ""))
    yoe       = float(p.get("years_of_experience", 0))
    feats     = {}

    # Hard disqualifiers
    feats["red_flag_title"] = 1.0 if any(rf in title_lc for rf in RED_FLAG_TITLES) else 0.0
    companies = [text_lower(j.get("company", "")) for j in history]
    consulting_count = sum(1 for co in companies if any(cf in co for cf in CONSULTING_FIRMS))
    feats["consulting_only"] = 1.0 if (len(companies) > 0 and consulting_count == len(companies)) else 0.0

    production_kws = ["production", "deployed", "shipped", "launched", "release",
                      "real users", "live", "scale", "serving", "inference"]
    feats["has_production_ml"] = min(1.0, keyword_hit_rate(full_text, production_kws) * 5)

    research_titles = ["research scientist", "research engineer", "phd intern",
                       "postdoc", "research associate", "academic"]
    research_count = sum(1 for j in history
                         if any(rt in text_lower(j.get("title", "")) for rt in research_titles))
    feats["research_heavy"] = min(1.0, research_count / max(len(history), 1))

    # Skills
    skill_names_lc = [text_lower(s.get("name", "")) for s in skills]
    feats["required_skills_hit"] = keyword_hit_rate(" ".join(skill_names_lc), REQUIRED_SKILLS)
    feats["nice_skills_hit"]     = keyword_hit_rate(full_text, NICE_TO_HAVE_SKILLS)
    expert_count = sum(1 for s in skills if s.get("proficiency") in ("expert", "advanced"))
    feats["expert_skill_count_norm"] = min(1.0, expert_count / 10)

    retrieval_terms = ["embedding", "vector", "retrieval", "faiss", "pinecone",
                       "weaviate", "qdrant", "milvus", "elasticsearch", "bm25",
                       "dense retrieval", "hybrid search", "sentence transformer",
                       "semantic search", "ann", "approximate nearest neighbor"]
    feats["retrieval_depth"] = min(1.0, keyword_hit_rate(full_text, retrieval_terms) * 3)

    eval_terms = ["ndcg", "mrr", "map", "a/b test", "ab test", "learning to rank",
                  "ltr", "xgboost rank", "ranknet", "lambdamart", "evaluation", "benchmark"]
    feats["eval_depth"] = min(1.0, keyword_hit_rate(full_text, eval_terms) * 4)

    llm_terms = ["llm", "rag", "fine-tun", "lora", "qlora", "peft", "instruction tuning",
                 "gpt", "bert", "transformers", "huggingface", "langchain"]
    feats["llm_depth"] = min(1.0, keyword_hit_rate(full_text, llm_terms) * 3)

    python_skills = [s for s in skills if "python" in text_lower(s.get("name", ""))]
    if python_skills:
        prof_map = {"expert": 1.0, "advanced": 0.8, "intermediate": 0.5, "beginner": 0.2}
        feats["python_strength"] = max(prof_map.get(s.get("proficiency", "beginner"), 0.1) for s in python_skills)
    else:
        feats["python_strength"] = 0.3 if "python" in full_text else 0.0

    # Experience
    feats["yoe_fit"] = yoe_score(yoe)
    ml_keywords = ["machine learning", "ml engineer", "ai engineer", "data scientist",
                   "nlp", "deep learning", "applied ml", "research engineer"]
    total_months = sum(j.get("duration_months", 0) for j in history)
    ml_months = sum(j.get("duration_months", 0) for j in history
                    if any(kw in text_lower(j.get("title", "") + " " + j.get("description", "")) for kw in ml_keywords))
    feats["ml_career_fraction"] = ml_months / max(total_months, 1)

    product_industries = ["technology", "software", "internet", "saas", "fintech",
                          "edtech", "healthtech", "e-commerce", "marketplace"]
    product_months = sum(j.get("duration_months", 0) for j in history
                         if any(pi in text_lower(j.get("industry", "")) for pi in product_industries))
    feats["product_company_fraction"] = product_months / max(total_months, 1)

    if len(history) >= 2:
        avg_tenure = total_months / len(history)
        feats["tenure_stability"] = min(1.0, avg_tenure / 36) if avg_tenure < 36 else max(0.5, 1.0 - (avg_tenure - 48) / 100)
    else:
        feats["tenure_stability"] = 0.5

    current_jobs = [j for j in history if j.get("is_current")]
    if current_jobs:
        curr = current_jobs[0]
        curr_ml = any(kw in text_lower(curr.get("title", "") + " " + curr.get("description", "")) for kw in ml_keywords)
        feats["current_role_ml"] = 1.0 if curr_ml else 0.2
    else:
        feats["current_role_ml"] = 0.3

    tier_map = {"tier_1": 1.0, "tier_2": 0.8, "tier_3": 0.6, "tier_4": 0.4, "unknown": 0.5}
    edu_score = max((tier_map.get(e.get("tier", "unknown"), 0.5) for e in edu), default=0.4)
    feats["edu_tier"] = edu_score

    # Behavioral
    last_active_days = days_since(signals.get("last_active_date", "2000-01-01"))
    if last_active_days < 7:     feats["recency"] = 1.0
    elif last_active_days < 30:  feats["recency"] = 0.85
    elif last_active_days < 90:  feats["recency"] = 0.6
    elif last_active_days < 180: feats["recency"] = 0.35
    else:                        feats["recency"] = 0.1

    feats["open_to_work"]         = 1.0 if signals.get("open_to_work_flag") else 0.4
    feats["recruiter_response"]   = float(signals.get("recruiter_response_rate", 0.0))
    notice = int(signals.get("notice_period_days", 90))
    if notice <= 15:   feats["notice_score"] = 1.0
    elif notice <= 30: feats["notice_score"] = 0.9
    elif notice <= 60: feats["notice_score"] = 0.6
    elif notice <= 90: feats["notice_score"] = 0.4
    else:              feats["notice_score"] = 0.2

    gh = float(signals.get("github_activity_score", -1))
    feats["github_score"]          = gh / 100 if gh >= 0 else 0.3
    feats["completeness"]          = float(signals.get("profile_completeness_score", 50)) / 100
    feats["recruiter_interest"]    = min(1.0, int(signals.get("saved_by_recruiters_30d", 0)) / 10)
    feats["interview_reliability"] = float(signals.get("interview_completion_rate", 0.5))

    pref = signals.get("preferred_work_mode", "flexible")
    feats["work_mode_fit"] = {"hybrid": 1.0, "flexible": 0.9, "onsite": 0.7, "remote": 0.5}.get(pref, 0.6)
    feats["location_fit"]  = location_score(c)

    sal = signals.get("expected_salary_range_inr_lpa", {})
    sal_mid = (float(sal.get("min", 0)) + float(sal.get("max", 999))) / 2
    if 20 <= sal_mid <= 70:   feats["salary_fit"] = 1.0
    elif 15 <= sal_mid < 20:  feats["salary_fit"] = 0.7
    elif 70 < sal_mid <= 100: feats["salary_fit"] = 0.6
    else:                     feats["salary_fit"] = 0.3

    feats["verified"]    = 1.0 if (signals.get("verified_email") and signals.get("verified_phone")) else 0.5
    feats["is_honeypot"] = 1.0 if cid in honeypot_ids else 0.0
    return feats

# ── Reasoning Generation ─────────────────────────────────────────────────────
def build_reasoning(c: dict, feats: dict, rank: int, score: float) -> str:
    p       = c.get("profile", {})
    signals = c.get("redrob_signals", {})
    skills  = c.get("skills", [])

    title   = p.get("current_title", "Engineer")
    yoe     = p.get("years_of_experience", 0)
    company = p.get("current_company", "")
    loc     = p.get("location", "")

    top_skills = sorted(
        [s for s in skills if s.get("proficiency") in ("expert", "advanced")],
        key=lambda s: s.get("endorsements", 0), reverse=True
    )[:3]
    top_skill_names = ", ".join(s["name"] for s in top_skills) if top_skills else "general ML"

    notice         = signals.get("notice_period_days", 90)
    rr             = signals.get("recruiter_response_rate", 0)
    last_active_days = days_since(signals.get("last_active_date", "2000-01-01"))

    strengths, concerns = [], []
    if feats.get("retrieval_depth", 0) > 0.5:     strengths.append("strong retrieval/embedding background")
    if feats.get("eval_depth", 0) > 0.3:          strengths.append("ranking evaluation experience (NDCG/MRR)")
    if feats.get("product_company_fraction", 0) > 0.6: strengths.append("product-company career")
    if feats.get("has_production_ml", 0) > 0.5:   strengths.append("production ML deployment")
    if feats.get("github_score", 0) > 0.5:        strengths.append("active open-source contributions")
    if feats.get("llm_depth", 0) > 0.4:           strengths.append("LLM/fine-tuning experience")

    if notice > 60:            concerns.append(f"long notice period ({notice}d)")
    if rr < 0.3:               concerns.append(f"low recruiter response rate ({rr:.0%})")
    if last_active_days > 90:  concerns.append(f"inactive for {last_active_days} days")
    if feats.get("red_flag_title", 0) > 0:  concerns.append(f"current title ({title}) is outside AI/ML domain")
    if feats.get("consulting_only", 0) > 0: concerns.append("career exclusively at consulting firms")
    if feats.get("research_heavy", 0) > 0.5: concerns.append("research-heavy background without clear production deployments")
    if feats.get("is_honeypot", 0) > 0:     concerns.append("profile has data inconsistencies (honeypot signal)")

    s1 = f"{yoe:.0f}-year {title} at {company}" if company else f"{yoe:.0f}-year {title}"
    if loc:         s1 += f", {loc}"
    if top_skills:  s1 += f"; top skills: {top_skill_names}"

    parts2 = []
    if rank <= 10:
        if strengths: parts2.append("Strong fit: " + ", ".join(strengths[:3]))
        if concerns:  parts2.append("Concern: " + "; ".join(concerns[:1]))
    elif rank <= 30:
        if strengths: parts2.append("Fit signals: " + ", ".join(strengths[:2]))
        if concerns:  parts2.append("Gap: " + "; ".join(concerns[:2]))
    elif rank <= 60:
        if strengths: parts2.append("Partial match: " + ", ".join(strengths[:1]))
        if concerns:  parts2.append("Notable gaps: " + "; ".join(concerns[:2]))
    else:
        if concerns:  parts2.append("Significant gaps: " + "; ".join(concerns[:3]))
        elif strengths: parts2.append("Adjacent skills only — weak JD fit overall")
        else:         parts2.append("Profile lacks core JD requirements")

    if not parts2:
        parts2.append("Limited alignment with JD requirements")
    return f"{s1}. {'; '.join(parts2)}."

# ── Validation ───────────────────────────────────────────────────────────────
def validate_submission(df: pd.DataFrame, all_candidate_ids: set, honeypot_ids: set) -> bool:
    errors, warnings_list = [], []

    if len(df) != 100:
        errors.append(f"Row count = {len(df)}, expected 100")

    ranks = sorted(df["rank"].tolist())
    if ranks != list(range(1, 101)):
        errors.append(f"Ranks not exactly 1-100 (got {len(set(ranks))} unique, min={min(ranks)}, max={max(ranks)})")

    if df["candidate_id"].nunique() != 100:
        errors.append("Duplicate candidate_ids detected")

    unknown = set(df["candidate_id"]) - all_candidate_ids
    if unknown:
        errors.append(f"Unknown candidate_ids: {unknown}")

    req_cols = ["candidate_id", "rank", "score", "reasoning"]
    for col in req_cols:
        if col not in df.columns:
            errors.append(f"Missing required column: {col}")
    if list(df.columns)[:4] != req_cols:
        errors.append(f"Column order wrong: got {list(df.columns)}, expected {req_cols}")

    sorted_by_rank = df.sort_values("rank")
    score_diffs = sorted_by_rank["score"].diff().dropna()
    if (score_diffs > 1e-9).any():
        errors.append(f"Scores not non-increasing: {(score_diffs > 1e-9).sum()} violation(s)")

    if df["score"].nunique() == 1:
        errors.append("All scores identical — model not differentiating")

    top10_honeypots = df[df["rank"] <= 10]["candidate_id"].apply(lambda x: x in honeypot_ids).sum()
    if top10_honeypots > 0:
        warnings_list.append(f"{top10_honeypots} honeypots in top 10 — risk of Stage 3 disqualification")

    if "reasoning" in df.columns:
        empty = df["reasoning"].isna().sum() + (df["reasoning"] == "").sum()
        if empty > 10:
            warnings_list.append(f"{empty} empty reasoning entries")
        if df["reasoning"].nunique() < 50:
            warnings_list.append(f"Low reasoning diversity: only {df['reasoning'].nunique()} unique values")

    if not errors:
        print("ALL VALIDATION CHECKS PASSED")
    else:
        print(f"{len(errors)} ERROR(S):")
        for e in errors:
            print(f"  * {e}")
    if warnings_list:
        print(f"{len(warnings_list)} WARNING(S):")
        for w in warnings_list:
            print(f"  * {w}")
    return len(errors) == 0

# ── Main Pipeline ────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(description="Redrob Hackathon Candidate Ranker")
    parser.add_argument("--candidates", required=True, help="Path to candidates.jsonl")
    parser.add_argument("--team-id",   default="NoBlackBox", help="Team ID (used for output filename)")
    parser.add_argument("--out",       default=None, help="Output CSV path (default: <team-id>.csv)")
    args = parser.parse_args()

    out_path = args.out or f"{args.team_id}.csv"
    t0 = time.time()

    # 1. Load candidates
    candidates = []
    errors = 0
    with open(args.candidates, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                c = json.loads(line)
                if "candidate_id" in c:
                    candidates.append(c)
                else:
                    errors += 1
            except json.JSONDecodeError:
                errors += 1
    print(f"Loaded {len(candidates):,} candidates ({errors} parse errors) in {time.time()-t0:.1f}s")

    # 2. Honeypot detection
    honeypot_ids = set()
    for c in tqdm(candidates, desc="Honeypot detection"):
        is_hp, _ = detect_honeypot(c, honeypot_ids)
        if is_hp:
            honeypot_ids.add(c["candidate_id"])
    print(f"Honeypots detected: {len(honeypot_ids)}")

    # 3. Feature extraction
    feature_rows, candidate_texts = [], []
    for c in tqdm(candidates, desc="Extracting features"):
        feats = extract_features(c, honeypot_ids)
        feats["candidate_id"] = c["candidate_id"]
        feature_rows.append(feats)
        candidate_texts.append(build_candidate_text(c))
    feat_df = pd.DataFrame(feature_rows).set_index("candidate_id")

    # 4. Semantic scoring (TF-IDF + BM25)
    tfidf = TfidfVectorizer(
        max_features=50_000, ngram_range=(1, 2), sublinear_tf=True,
        min_df=2, strip_accents="unicode", analyzer="word",
        token_pattern=r"(?u)\b[a-z][a-z+#/\.]{1,}\b"
    )
    candidate_tfidf = tfidf.fit_transform(candidate_texts)
    jd_vec = tfidf.transform([JD_TEXT])
    tfidf_scores = cosine_similarity(jd_vec, candidate_tfidf).flatten()

    tokenized = [text.lower().split() for text in candidate_texts]
    bm25 = BM25Okapi(tokenized)
    bm25_scores_raw = bm25.get_scores(JD_TEXT.lower().split())
    bm25_max = bm25_scores_raw.max()
    bm25_norm = bm25_scores_raw / bm25_max if bm25_max > 0 else bm25_scores_raw
    semantic_scores = 0.6 * tfidf_scores + 0.4 * bm25_norm

    cand_ids = [c["candidate_id"] for c in candidates]
    semantic_df = pd.DataFrame({
        "candidate_id": cand_ids,
        "tfidf_score":  tfidf_scores,
        "bm25_score":   bm25_norm,
        "semantic_score": semantic_scores
    }).set_index("candidate_id")

    # 5. Composite score
    all_df = feat_df.join(semantic_df, how="left")
    raw_score = sum(all_df[col].fillna(0) * w for col, w in WEIGHTS.items() if col in all_df.columns)
    penalty_mult = pd.Series(1.0, index=all_df.index)
    for pen_col, pen_factor in PENALTIES.items():
        if pen_col in all_df.columns:
            penalty_mult *= (1 - all_df[pen_col].fillna(0) * pen_factor)
    composite = raw_score * penalty_mult
    c_min, c_max = composite.min(), composite.max()
    composite_norm = (composite - c_min) / (c_max - c_min) if c_max > c_min else composite.copy()
    all_df["composite_score"] = composite_norm

    ranked = all_df.sort_values(["composite_score", "candidate_id"], ascending=[False, True]).reset_index()
    top100 = ranked.head(100).copy()
    top100["rank"] = range(1, 101)

    # 6. Reasoning
    cand_map = {c["candidate_id"]: c for c in candidates}
    feat_map = {row["candidate_id"]: row.to_dict() for _, row in feat_df.reset_index().iterrows()}
    reasonings = []
    for _, row in tqdm(top100.iterrows(), total=100, desc="Generating reasoning"):
        cid   = row["candidate_id"]
        r = build_reasoning(cand_map.get(cid, {}), feat_map.get(cid, {}), int(row["rank"]), float(row["composite_score"]))
        reasonings.append(r)
    top100["reasoning"] = reasonings

    # 7. Build submission DataFrame — exact column order per spec
    submission = top100[["candidate_id", "rank", "composite_score", "reasoning"]].copy()
    submission = submission.rename(columns={"composite_score": "score"})
    submission = submission.sort_values("rank").reset_index(drop=True)

    # Enforce non-increasing scores (fix floating-point ties)
    scores = submission["score"].values.copy()
    for i in range(1, len(scores)):
        if scores[i] > scores[i - 1]:
            scores[i] = scores[i - 1]
    submission["score"] = scores

    # 8. Validate
    all_ids = {c["candidate_id"] for c in candidates}
    is_valid = validate_submission(submission, all_ids, honeypot_ids)

    # 9. Save
    if is_valid:
        submission.to_csv(out_path, index=False, encoding="utf-8")
        print(f"Submission saved to: {out_path}")
        print(f"Rows: {len(submission)} | Score range: {submission['score'].min():.4f} - {submission['score'].max():.4f}")
    else:
        print("Submission NOT saved — fix errors above first.")

    total_time = time.time() - t0
    print(f"Total pipeline time: {total_time:.1f}s ({total_time/60:.2f} min)")
    if total_time >= 300:
        print("WARNING: Exceeded 5-minute compute budget!")

if __name__ == "__main__":
    main()
'''

with open("rank.py", "w") as f:
    f.write(RANK_SCRIPT)
print("rank.py written successfully — full pipeline included.")

REQUIREMENTS = """rank_bm25>=0.2.2
scikit-learn>=1.3.0
numpy>=1.24.0
pandas>=2.0.0
tqdm>=4.65.0
"""

with open("requirements.txt", "w") as f:
    f.write(REQUIREMENTS)
print("requirements.txt written.")

rank.py written successfully — full pipeline included.
requirements.txt written.


#### Author: Mahtab Madni